# 10 · Difusión DDIM condicional

Entrena un modelo de difusión 1D condicionado al régimen y muestrea con DDIM determinista y guiado sin clasificador.

**Responsable:** Daniel

**Entradas**

- `data/processed/ventanas.npz`

**Salidas**

- `models/generadores/difusion/ (modelo.pkl o .keras, historial.csv, meta.json)`
- `data/synthetic/difusion.npz`

**Tiempo estimado:** ~40 min en CPU (200 épocas de entrenamiento más el muestreo DDIM del banco).

**Independencia.** Este notebook solo lee `data/processed/ventanas.npz` (notebook 02) y solo escribe en `models/generadores/difusion/` y `data/synthetic/difusion.npz`. No depende de ningún otro notebook de generador ni de sus salidas, de modo que los notebooks 04 a 10 pueden ejecutarse en paralelo y en cualquier orden por distintas personas.

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import ventanas

part = ventanas.cargar_procesado()
train, val, test = part.train, part.val, part.test
print(train, val, test, sep="\n")

In [ ]:
from src import regimenes
from src.generadores import base

v = config.ventanas()
n_regimenes = config.n_regimenes()

bloque_train = ventanas.empaquetar(train)
print("bloque de train:", bloque_train.shape, "· d esperada:", ventanas.dimension_bloque(v))
regimenes.distribucion(train.y_reg, n_regimenes)

## Diseño del modelo de difusión

El proceso directo añade ruido gaussiano al bloque en 1.000 pasos con planificador
coseno; la red aprende a predecir el ruido añadido. El planificador coseno en lugar
del lineal reparte mejor la señal: el lineal destruye casi toda la información en el
primer tercio del proceso y desperdicia capacidad en pasos donde no queda nada que
predecir.

**Muestreo DDIM y no DDPM.** DDIM es determinista y permite muestrear con 50 pasos
en vez de los 1.000 del proceso entrenado. Con un banco de miles de muestras en CPU,
esa diferencia es la que separa lo viable de lo inviable.

**Guiado sin clasificador.** Durante el entrenamiento la condición se sustituye por
un token nulo con probabilidad 0.1, lo que deja al mismo modelo aprendiendo la
versión condicional y la incondicional. Al muestrear se extrapola entre ambas con un
factor de guiado, que agudiza la dependencia del régimen. Es exactamente el
mecanismo que hace útiles a estos generadores en el taller: sube la fidelidad a la
condición *crisis* a costa de algo de diversidad.

**Reducción de dimensión.** Se trabaja sobre las primeras componentes principales del
bloque, no sobre las 1.201 dimensiones crudas. Con 1.201 entradas y unos miles de
ventanas la red dedica la mayor parte de su capacidad a direcciones de varianza
despreciable, y en CPU el coste por época no lo justifica.

In [ ]:
generador = base.instanciar(
    "difusion",
    n_regimenes=n_regimenes,
    pasos_difusion=1000,
    planificador="coseno",
    n_pasos_muestreo=50,
    guiado=1.5,
    prob_dropout_condicion=0.1,
    ancho=384,
    n_bloques=3,
    epocas=200,
    tam_lote=128,
    n_componentes=256,
    verboso=1,
)
generador.fit(bloque_train, train.y_reg)
generador

## Convergencia

La pérdida es el MSE de predicción del ruido, promediado sobre pasos de difusión
muestreados al azar. Debe bajar rápido en las primeras épocas y luego aplanarse en
una meseta claramente por encima de cero: predecir el ruido en pasos muy avanzados
es imposible por construcción, y esa parte de la pérdida es irreducible.

Lo que hay que descartar aquí es una curva que siga bajando al final —presupuesto
corto— o que oscile con amplitud creciente —tasa de aprendizaje demasiado alta.

In [ ]:
fig, eje = plt.subplots()
viz.curva_convergencia(generador.historial, "Convergencia · " + generador.etiqueta, eje=eje)
eje.set_yscale("log")
viz.guardar(fig, "convergencia_difusion")

generador.historial.tail(3).round(4)

## Inspección visual

Proyección PCA de reales y sintéticos, con la PCA ajustada **solo con los reales**
para que los ejes describan la estructura del mercado y no la del generador.

Es la comprobación más rápida y la que detecta los dos fallos gruesos: si la nube
sintética no cubre la real, el generador ha colapsado a un modo; si la desborda
ampliamente, está inventando configuraciones de mercado que nunca ocurrieron.

Se mira el régimen de crisis porque es el que tiene menos datos reales y, por
tanto, donde el generador tiene más margen para desviarse.

In [ ]:
CRISIS = n_regimenes - 1

muestra_crisis = generador.generate(600, regimen=CRISIS)
reales_crisis = bloque_train[train.y_reg == CRISIS]

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
viz.real_vs_sintetico(bloque_train, generador.generate(600, regimen=0),
                      "{} · régimen de calma".format(generador.etiqueta), eje=ejes[0])
viz.real_vs_sintetico(reales_crisis, muestra_crisis,
                      "{} · régimen de crisis".format(generador.etiqueta), eje=ejes[1])
fig.tight_layout()
viz.guardar(fig, "pca_" + generador.nombre)

print("reales de crisis:", len(reales_crisis), "· sintéticos generados:", len(muestra_crisis))

## Banco de muestras

Se genera un banco uniforme por régimen y se exporta a `data/synthetic/`. La mezcla
concreta de cada dataset la decide el notebook 11 muestreando de este banco, no
volviendo a invocar al generador: así el barrido no necesita tener los siete
modelos cargados en memoria y dos ejecuciones del notebook 12 usan exactamente las
mismas muestras sintéticas.

El banco es uniforme —no replica el desbalance real— porque la política de reparto
es un grado de libertad del experimento y se aplica después.

In [ ]:
MUESTRAS_POR_REGIMEN = 2000

reparto = {k: MUESTRAS_POR_REGIMEN for k in range(n_regimenes)}
bloques_sint, y_sint = generador.generate_dataset(reparto)

print("banco:", bloques_sint.shape, "· etiquetas:", np.bincount(y_sint, minlength=n_regimenes))
print("rango de valores:", round(float(bloques_sint.min()), 2), "→",
      round(float(bloques_sint.max()), 2),
      "(referencia real:", round(float(bloque_train.min()), 2), "→",
      round(float(bloque_train.max()), 2), ")")

## Persistencia

`guardar()` deja el modelo, la curva de convergencia y los metadatos en
`models/generadores/`. Es lo que permite que el resto del grupo salte directamente
al análisis sin reentrenar nada.

In [ ]:
ruta_muestras = generador.exportar_muestras(bloques_sint, y_sint)
ruta_modelo = generador.guardar()

print("muestras:", ruta_muestras)
print("modelo:  ", ruta_modelo)
pd.Series(generador.resumen_convergencia()).round(4)

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_MODELOS_GEN / "difusion" / "meta.json",
    src.DIR_MODELOS_GEN / "difusion" / "historial.csv",
    src.DIR_SINTETICO / "difusion.npz",
    src.DIR_FIGURAS / "convergencia_difusion.png",
    src.DIR_FIGURAS / "pca_difusion.png",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
